In [16]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score
import os
from tensorflow.keras import layers, applications

In [17]:
MODEL1_DIR = '/kaggle/input/densenet121-ensemble/DenseNet121_Ensemble'
preprocess_fn1 = applications.densenet.preprocess_input

MODEL2_DIR = '/kaggle/input/resnet50-ensemble/ResNet50_Ensemble'
preprocess_fn2 = applications.resnet.preprocess_input

MODEL3_DIR = '/kaggle/input/vgg19-ensemble/VGG19_Ensemble'
preprocess_fn3 = applications.vgg19.preprocess_input

MODEL4_DIR = '/kaggle/input/xception-ensemble/Xception_Ensemble'
preprocess_fn4 = applications.xception.preprocess_input

MODEL5_DIR = '/kaggle/input/mobilenet-ensemble/MobileNet_Ensemble'
preprocess_fn5 = applications.mobilenet.preprocess_input

In [18]:
TRAIN_DIR = '/kaggle/input/5-fold-brain-tumor-contrast-enhanced/kfold_dataset'

In [19]:
def tta_augmentation(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.1)
    img = tf.image.random_contrast(img, lower=0.9, upper=1.1)
    return img

In [20]:
def tta_predict(model, preprocess_fn, ds, tta_rounds=5):
    AUTOTUNE = tf.data.AUTOTUNE
    
    val_ds = ds.map(lambda image, label: (preprocess_fn(image), label)).prefetch(AUTOTUNE)
    preds = [model.predict(val_ds)]

    for _ in range(tta_rounds):
        tta_ds = ds.map(lambda image, label: (tta_augmentation(image), label)).prefetch(AUTOTUNE)
        tta_ds = tta_ds.map(lambda image, label: (preprocess_fn(image), label)).prefetch(AUTOTUNE)
        preds.append(model.predict(tta_ds))

    preds = np.array(preds)
    return np.mean(preds, axis=0)

In [21]:
def get_pred_for_ensembling_model(fold_k, val_ds):
    model1 = tf.keras.models.load_model(os.path.join(MODEL1_DIR, f'DenseNet121_block_2_fold_{fold_k}.keras'), compile=False)
    # model2 = tf.keras.models.load_model(os.path.join(MODEL2_DIR, f'ResNet50_block_1_fold_{fold_k}.keras'), compile=False)
    # model3 = tf.keras.models.load_model(os.path.join(MODEL3_DIR, f'VGG19_block_1_fold_{fold_k}.keras'), compile=False)
    model4 = tf.keras.models.load_model(os.path.join(MODEL4_DIR, f'Xception_block_2_fold_{fold_k}.keras'), compile=False)
    # model5 = tf.keras.models.load_model(os.path.join(MODEL5_DIR, f'MobileNet_block_1_fold_{fold_k}.keras'), compile=False)

    y_true = np.concatenate([y.numpy() for _, y in val_ds], axis=0)
    pred1 = tta_predict(model1, preprocess_fn1, val_ds)
    # pred2 = tta_predict(model2, preprocess_fn2, val_ds)
    # pred3 = tta_predict(model3, preprocess_fn3, val_ds)
    pred4 = tta_predict(model4, preprocess_fn4, val_ds)
    # pred5 = tta_predict(model5, preprocess_fn5, val_ds)
    ensemble_pred = (pred1 + pred4) / 2.0

    return ensemble_pred

In [22]:
DATASET_CACHE = {}
def get_validation_fold(k, train_dir=TRAIN_DIR, IMG_SIZE=(224, 224), BATCH_SIZE=32, SEED=24520152):
    val_dir = os.path.join(train_dir, f'Subset_{k}')
    
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
        seed=SEED,
    )
    
    AUTOTUNE = tf.data.AUTOTUNE

    return val_ds

In [23]:
def get_predictions_for_fold(k):
    valid_data = get_validation_fold(k)

    y_true = []
    for _, y in valid_data:
        y_true.extend(y.numpy())
    y_true = np.array(y_true)
    y_pred = []

    y_pred.append(get_pred_for_ensembling_model(k, valid_data))
    
    y_pred = np.array(y_pred)
    return y_true, y_pred

In [24]:
def macro_specificity(y_true, y_pred, num_classes):
    cm = confusion_matrix(y_true, y_pred)
    spec = []

    for i in range(num_classes):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - (TP + FN + FP)

        spec.append(TN / (TN + FP + 1e-8))

    return np.array(spec)

In [25]:
def evaluate_fold(k):
    y_true, y_pred = get_predictions_for_fold(k)

    model_results = []
    for i in range(len(y_pred)):
        y_pred_label = np.argmax(y_pred[i], axis=1)
        acc = accuracy_score(y_true, y_pred_label)
        precision = precision_score(y_true, y_pred_label, average='macro', zero_division=0)
        recall = recall_score(y_true, y_pred_label, average='macro', zero_division=0)
        f1 = f1_score(y_true, y_pred_label, average='macro', zero_division=0)
        spec_per_class = macro_specificity(y_true, y_pred_label, 3)
        specificity = np.mean(spec_per_class)

        model_results.append({
            'Accuracy': np.array(acc),
            'Precision': np.array(precision),
            'Recall': np.array(recall),
            'F1-Score': np.array(f1),
            'Specificity': np.array(specificity)
        })
    return np.array(model_results)

In [26]:
num_run = 5
num_fold = 5
num_model = 1
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity']
all_result = []

for i in range(num_run):
    for j in range(1, 6):
        result = evaluate_fold(j)
        all_result.append(result)

all_result = np.array(all_result)
avg_metrics = {}

for m in range(num_model):
    avg_metrics[m] = {}
    for metric in metric_names:
        values = np.array([
            all_result[k][m][metric] for k in range(len(all_result))
        ])
        avg_metrics[m][metric] = values.mean(axis=0)*100

Found 542 files belonging to 3 classes.
17/17 ━━━━━━━━━━━━━━━━━━━━ 30s 973ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 88ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 92ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 9s 298ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 84ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 78ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 80ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 80ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 78ms/step
Found 679 files belonging to 3 classes.
22/22 ━━━━━━━━━━━━━━━━━━━━ 26s 725ms/step
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 86ms/step
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 82ms/step
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 84ms/step
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step
22/22 ━━━━━━━━━━━━━━━━━━━━ 9s 245ms/step
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 78ms/step
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms/step
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 78ms/step
22/22 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms

In [27]:
row_names = ['tta-densenet121-xception']
column_names = ['Recall', 'Specificity', 'Precision', 'F1-Score', 'Accuracy']

df_result = pd.DataFrame(avg_metrics).T
df_result.index = row_names
df_result = df_result[column_names]
df_result = df_result.round(2)
df_result

,Recall,Specificity,Precision,F1-Score,Accuracy
tta-densenet121-xception,91.6,96.28,91.68,91.43,92.53


In [28]:
latex_table = df_result.to_latex(
    multicolumn=True,
    multirow=True,
    float_format="%.2f",
    label="tab:sens_spec"
)

print(latex_table)

\begin{table}
\label{tab:sens_spec}
\begin{tabular}{lrrrrr}
\toprule
 & Recall & Specificity & Precision & F1-Score & Accuracy \\
\midrule
tta-densenet121-xception & 91.60 & 96.28 & 91.68 & 91.43 & 92.53 \\
\bottomrule
\end{tabular}
\end{table}

